# NABirds Overnight Runner
This notebook is a standalone multi-run trainer for NABirds ResNet experiments.

It includes:
- dataset preparation (species merge + bbox crop + 240x240 padding),
- Stage 1 (head-only) and optional Stage 2 (layer4 + fc) training,
- repeatable run programs,
- CSV/JSONL logs in `artifacts/logs/`,
- result summary tables.


In [85]:
from pathlib import Path
import os
import re
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None



In [86]:
# Paths and reproducibility
DATA_ROOT = Path("NABirds_Dataset/nabirds")
IMAGES_DIR = DATA_ROOT / "images"

SEED = 42
VAL_FRACTION = 0.10
BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 4

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

# Notebook + spawn multiprocessing often fails to pickle custom Dataset classes.
NUM_WORKERS = 0 if IN_NOTEBOOK else DEFAULT_NUM_WORKERS
NUM_EPOCHS = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
TARGET_SIZE = 240
LABEL_SMOOTHING = 0.00

# Split file config
DEFAULT_TRAIN_TEST_SPLIT_FILE = "train_test_split.txt"
TRAIN_TEST_SPLIT_FILE = "train_test_split_8020_target_species.txt"
#TRAIN_TEST_SPLIT_FILE = "train_test_split_8020_all_specific.txt"


# Defaults used for checkpoint naming tags
DEFAULT_WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
print(f"IN_NOTEBOOK={IN_NOTEBOOK}, NUM_WORKERS={NUM_WORKERS}")


Using device: mps
IN_NOTEBOOK=True, NUM_WORKERS=0


In [87]:
# Target species list (from your original notebook)
TARGET_SPECIES = [
    "American Robin",
    "Mourning Dove",
    "Blue Jay",
    "House Sparrow",
    "Canada Goose",
    "Northern Mockingbird",
    "American Crow",
    "Carolina Chickadee",
    "Red-tailed Hawk",
    "Northern Cardinal",
    "Killdeer",
    "Dark-eyed Junco",
    "White-breasted Nuthatch",
    "Bald Eagle",
    "Red-bellied Woodpecker",
    "Carolina Wren",
    "Song Sparrow",
    "Wild Turkey",
    "European Starling",
    "Tufted Titmouse",
    "Mallard",
    "Common Raven",
    "American Goldfinch",
    "House Wren",
    "Eastern Phoebe",
    "Common Yellowthroat",
    "Great Horned Owl",
    "Northern Flicker",
    "Grey Catbird",
    "White-throated Sparrow",
    "Chimney Swift",
    "Belted Kingfisher",
    "Red-winged Blackbird",
    "Laughing Gull",
    "House Finch",
    "Common Loon",
    "Great Crested Flycatcher",
    "Ruby-crowned Kinglet",
    "Hermit Thrush",
    "Wood Duck",
    "Yellow Warbler",
    "Chipping Sparrow",
    "Red-eyed Vireo",
    "Tree Swallow",
    "Cooper's Hawk",
    "Ovenbird",
    "Winter Wren",
    "Cedar Waxwing",
    "Snow Goose",
    "Brown Thrasher",
    "Eastern Screech Owl",
    "Downy Woodpecker",
    "Rock Pigeon",
    "Wood Thrush",
    "Red-breasted Nuthatch",
    "Common Grackle",
    "Eastern Wood-Pewee",
    "Pileated Woodpecker",
    "Brown-headed Cowbird",
    "American Woodcock",
    "Barred Owl",
    "Golden-crowned Kinglet",
    "Eastern Whip-poor-will",
    "Indigo Bunting",
    "Brown Creeper",
    "Fish Crow",
    "Barn Swallow",
    "Eastern Towhee",
    "Warbling Vireo",
    "Ruby-throated Hummingbird",
    "Field Sparrow",
    "Eastern Bluebird",
    "Hairy Woodpecker",
    "Baltimore Orioles",
    "Eastern Meadowlark",
    "Black-capped Chickadee",
    "Osprey",
    "Scarlet Tanager",
    "Eastern Kingbird",
    "Great Blue Heron",
    "Yellow-billed Cuckoo",
    "Red-headed Woodpecker",
    "Rose-breasted Grosbeak",
    "Yellow-bellied Sapsucker",
    "Black-and-white Warbler",
    "Willow Flycatcher",
    "Hooded Merganser",
    "American Redstart",
    "Green Heron",
    "Purple Martin",
    "Yellow-rumped Warbler",
    "American Kestrel",
    "Common Nighthawk",
    "Ruffed Grouse",
    "Common Merganser",
    "Great Egret",
    "Double-crested Cormorant",
    "Mute Swan",
    "Turkey Vulture",
    "Black Vulture",
]


# Optional: override TARGET_SPECIES from a CSV with a `species` column.
# Example: "artifacts/label_names_nabirds_all_specific.csv"
TARGET_SPECIES_CSV_OVERRIDE = False #"artifacts/label_names_nabirds_all_specific.csv"
if TARGET_SPECIES_CSV_OVERRIDE:
    override_path = Path(TARGET_SPECIES_CSV_OVERRIDE)
    if not override_path.exists():
        raise FileNotFoundError(f"TARGET_SPECIES_CSV_OVERRIDE not found: {override_path}")
    override_df = pd.read_csv(override_path)
    if "species" not in override_df.columns:
        raise ValueError("Override CSV must contain a `species` column")
    TARGET_SPECIES = override_df["species"].dropna().astype(str).tolist()

len(TARGET_SPECIES)


100

## Experiment Controls (Edit First)
Set your run program, split file, and species override in the cells above this point before running the notebook.


In [88]:
# Configure experiment runs here (edit this near the top before executing).
# Add one config dict per experiment; `repeats` controls repeated runs for that config.
RUN_PROGRAM = [
    {
        "name": "learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/2,
        "weight_decay": 2*WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/2,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-half_label-smooth-0-05_decay-1e4_s2-lr-half",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/2,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/2,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-quarter",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/2,
        "weight_decay": 2*WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/4,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-half_label-smooth-0-05_decay-1e4_s2-lr-quarter",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/2,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/4,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },

    {
        "name": "learning-rate-quarter_label-smooth-0-05_decay-2e4_s2-lr-half",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/4,
        "weight_decay": 2*WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/2,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-half",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/4,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/2,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-quarter_label-smooth-0-05_decay-2e4_s2-lr-quarter",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/4,
        "weight_decay": 2*WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/4,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
    {
        "name": "learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter",
        "repeats": 1,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "seed": SEED,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE/4,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": 0.05,
        "batch_size": BATCH_SIZE,
        "run_stage2": True,
        "stage2_epochs": STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5,
        "stage2_lr": STAGE2_LR if 'STAGE2_LR' in globals() else (1e-4)/4,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY,
    },
]

print("RUN_PROGRAM ready. Config count:", len(RUN_PROGRAM))
for cfg in RUN_PROGRAM:
    print("-", cfg.get("name", "exp"), "repeats=", cfg.get("repeats", 1), "split=", cfg.get("split_file", TRAIN_TEST_SPLIT_FILE))


RUN_PROGRAM ready. Config count: 8
- learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-half_label-smooth-0-05_decay-1e4_s2-lr-half repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-quarter repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-half_label-smooth-0-05_decay-1e4_s2-lr-quarter repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-quarter_label-smooth-0-05_decay-2e4_s2-lr-half repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-half repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-quarter_label-smooth-0-05_decay-2e4_s2-lr-quarter repeats= 1 split= train_test_split_8020_target_species.txt
- learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter repeats= 1 split= train_test_split_8020_target_s

In [89]:
def canonicalize_name(name: str) -> str:
    # Remove variant suffixes like "(Adult male)", then normalize punctuation and spacing.
    name = re.sub(r"\s*\([^)]*\)\s*", " ", name)
    name = name.lower()
    name = name.replace("grey", "gray")
    name = name.replace("orioles", "oriole")
    name = name.replace("-", " ")
    name = name.replace("'", "")
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Load NABirds metadata
images = pd.read_csv(DATA_ROOT / "images.txt", sep=" ", names=["image_id", "image_rel_path"])
labels = pd.read_csv(DATA_ROOT / "image_class_labels.txt", sep=" ", names=["image_id", "class_id"])
splits = pd.read_csv(DATA_ROOT / TRAIN_TEST_SPLIT_FILE, sep=" ", names=["image_id", "is_train"])
bboxes = pd.read_csv(DATA_ROOT / "bounding_boxes.txt", sep=" ", names=["image_id", "x", "y", "w", "h"])
# Parse classes.txt robustly: first token is class_id, remainder is class_name.
class_rows = []
with open(DATA_ROOT / "classes.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid, cname = line.split(maxsplit=1)
        class_rows.append((int(cid), cname))
classes = pd.DataFrame(class_rows, columns=["class_id", "class_name"])

valid_class_ids = set(labels["class_id"].unique())
classes = classes[classes["class_id"].isin(valid_class_ids)].copy()
classes["canon"] = classes["class_name"].map(canonicalize_name)

species_to_class_ids = {}
dropped_species = []
normalization_report = {}

for species in TARGET_SPECIES:
    canon = canonicalize_name(species)
    matched = classes.loc[classes["canon"] == canon, "class_id"].tolist()
    if matched:
        species_to_class_ids[species] = sorted(set(matched))
        # Show auto-normalized cases for transparency.
        canonical_forms = sorted(set(classes.loc[classes["canon"] == canon, "class_name"]))
        if species not in canonical_forms:
            normalization_report[species] = canonical_forms
    else:
        dropped_species.append(species)

print(f"Requested species: {len(TARGET_SPECIES)}")
print(f"Mapped species:    {len(species_to_class_ids)}")
print(f"Dropped species:   {len(dropped_species)}")
if dropped_species:
    print("Dropped (not in NABirds):", dropped_species)

if normalization_report:
    print("\nAuto-normalized names:")
    for src, matched_names in normalization_report.items():
        print(f"- {src} -> {matched_names[0]}")


Requested species: 100
Mapped species:    98
Dropped species:   2
Dropped (not in NABirds): ['Eastern Whip-poor-will', 'Willow Flycatcher']

Auto-normalized names:
- American Robin -> American Robin (Adult)
- House Sparrow -> House Sparrow (Female/Juvenile)
- Red-tailed Hawk -> Red-tailed Hawk (Dark morph)
- Northern Cardinal -> Northern Cardinal (Adult Male)
- Dark-eyed Junco -> Dark-eyed Junco (Oregon)
- Bald Eagle -> Bald Eagle (Adult, subadult)
- European Starling -> European Starling (Breeding Adult)
- Mallard -> Mallard (Breeding male)
- American Goldfinch -> American Goldfinch (Breeding Male)
- Common Yellowthroat -> Common Yellowthroat (Adult Male)
- Northern Flicker -> Northern Flicker (Red-shafted)
- Grey Catbird -> Gray Catbird
- White-throated Sparrow -> White-throated Sparrow (Tan-striped/immature)
- Red-winged Blackbird -> Red-winged Blackbird (Female/juvenile)
- Laughing Gull -> Laughing Gull (Breeding)
- House Finch -> House Finch (Adult Male)
- Common Loon -> Common Lo

In [90]:
# Build a class_id -> final species index map (merging all variants into one label)
final_species = sorted(species_to_class_ids.keys())
species_to_idx = {name: i for i, name in enumerate(final_species)}

class_id_to_species_idx = {}
for species, class_ids in species_to_class_ids.items():
    y = species_to_idx[species]
    for cid in class_ids:
        class_id_to_species_idx[cid] = y

print(f"Final class count: {len(final_species)}")


Final class count: 98


In [91]:
df = images.merge(labels, on="image_id", how="inner")
df = df.merge(splits, on="image_id", how="inner")
df = df.merge(bboxes, on="image_id", how="inner")

df = df[df["class_id"].isin(class_id_to_species_idx)].copy()
df["target"] = df["class_id"].map(class_id_to_species_idx)
df["species"] = df["target"].map({v: k for k, v in species_to_idx.items()})
df["image_path"] = df["image_rel_path"].map(lambda p: str(IMAGES_DIR / p))

print("Filtered samples:", len(df))
print("Train/Test by official split:")
print(df["is_train"].value_counts().rename(index={1: "train", 0: "test"}))


Filtered samples: 14579
Train/Test by official split:
is_train
train    11663
test      2916
Name: count, dtype: int64


In [92]:
# Stratified validation split from official train split
# Normalize split dtype defensively (some pandas versions parse as object/string).
df = df.copy()
df["is_train"] = pd.to_numeric(df["is_train"], errors="coerce")
df = df[df["is_train"].isin([0, 1])].copy()
df["is_train"] = df["is_train"].astype(int)

train_df = df[df["is_train"] == 1].copy().reset_index(drop=True)
test_df = df[df["is_train"] == 0].copy().reset_index(drop=True)

if train_df.empty:
    raise RuntimeError("No training samples found after filtering. Check earlier mapping output and class matches.")

rng = np.random.default_rng(SEED)
val_indices = []

for _, group in train_df.groupby("target", sort=False):
    idxs = group.index.to_numpy(copy=True)
    rng.shuffle(idxs)

    if len(idxs) < 2:
        n_val = 0
    else:
        n_val = max(1, int(round(len(idxs) * VAL_FRACTION)))
        n_val = min(n_val, len(idxs) - 1)

    if n_val > 0:
        val_indices.extend(idxs[:n_val].tolist())

# Fallback: guarantee non-empty val split when possible.
if len(val_indices) == 0 and len(train_df) > 1:
    val_indices = [int(rng.integers(0, len(train_df)))]

val_mask = train_df.index.isin(val_indices)
val_df = train_df[val_mask].copy().reset_index(drop=True)
train_df = train_df[~val_mask].copy().reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")


Train samples: 10487
Val samples:   1176
Test samples:  2916


In [93]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGENET_PAD_RGB = tuple(int(round(c * 255)) for c in IMAGENET_MEAN)


def crop_resize_pad(img: Image.Image, bbox, size=240, pad_rgb=(124, 116, 104)) -> Image.Image:
    x, y, w, h = bbox
    x1 = max(0, int(np.floor(x)))
    y1 = max(0, int(np.floor(y)))
    x2 = min(img.width, int(np.ceil(x + w)))
    y2 = min(img.height, int(np.ceil(y + h)))

    if x2 <= x1 or y2 <= y1:
        cropped = img
    else:
        cropped = img.crop((x1, y1, x2, y2))

    scale = min(size / cropped.width, size / cropped.height)
    new_w = max(1, int(round(cropped.width * scale)))
    new_h = max(1, int(round(cropped.height * scale)))
    resized = cropped.resize((new_w, new_h), resample=Image.BILINEAR)

    pad_left = (size - new_w) // 2
    pad_top = (size - new_h) // 2
    pad_right = size - new_w - pad_left
    pad_bottom = size - new_h - pad_top

    return ImageOps.expand(resized, border=(pad_left, pad_top, pad_right, pad_bottom), fill=pad_rgb)


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class NABirdsSpeciesDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.df = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img = crop_resize_pad(
            img,
            bbox=(row["x"], row["y"], row["w"], row["h"]),
            size=TARGET_SIZE,
            pad_rgb=IMAGENET_PAD_RGB,
        )

        if self.transform is not None:
            img = self.transform(img)

        target = int(row["target"])
        return img, target


In [94]:
# Datasets (global defaults; per-run loaders are built in the run manager)
train_ds = NABirdsSpeciesDataset(train_df, transform=train_transform)
val_ds = NABirdsSpeciesDataset(val_df, transform=eval_transform)
test_ds = NABirdsSpeciesDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Datasets ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


Datasets ready: train=10487, val=1176, test=2916


In [95]:
# Stage defaults used by the overnight manager
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STAGE2_EPOCHS = 5
STAGE2_LR = 1e-4
STAGE2_WEIGHT_DECAY = WEIGHT_DECAY


In [96]:
def fmt_gb(num_bytes: int) -> str:
    return f"{num_bytes / (1024 ** 3):.2f} GB"


def get_device_memory_label(device: torch.device) -> str:
    if device.type == "cuda":
        allocated = torch.cuda.memory_allocated(device)
        reserved = torch.cuda.memory_reserved(device)
        return f"alloc={fmt_gb(allocated)} reserved={fmt_gb(reserved)}"
    if device.type == "mps" and hasattr(torch, "mps"):
        alloc = torch.mps.current_allocated_memory()
        driver = torch.mps.driver_allocated_memory()
        rec_max = torch.mps.recommended_max_memory()
        return f"alloc={fmt_gb(alloc)} driver={fmt_gb(driver)} max={fmt_gb(rec_max)}"
    return "n/a"


def run_epoch(model, loader, criterion, optimizer=None, split_name="train"):
    is_train = optimizer is not None
    model.train(is_train)

    proc = psutil.Process(os.getpid()) if psutil is not None else None

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    epoch_start = time.perf_counter()
    pbar = tqdm(loader, leave=False, desc=f"{split_name}")

    for images, targets in pbar:
        step_start = time.perf_counter()

        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, targets)
            if is_train:
                loss.backward()
                optimizer.step()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        elif DEVICE.type == "mps" and hasattr(torch, "mps"):
            torch.mps.synchronize()

        preds = logits.argmax(dim=1)
        running_loss += loss.item() * targets.size(0)
        running_correct += (preds == targets).sum().item()
        running_total += targets.size(0)

        step_time = time.perf_counter() - step_start
        avg_loss_so_far = running_loss / max(1, running_total)
        avg_acc_so_far = running_correct / max(1, running_total)

        rss_label = fmt_gb(proc.memory_info().rss) if proc is not None else "n/a"
        device_mem_label = get_device_memory_label(DEVICE)

        pbar.set_postfix(
            loss=f"{avg_loss_so_far:.4f}",
            acc=f"{avg_acc_so_far:.4f}",
            step_s=f"{step_time:.3f}",
            rss=rss_label,
            dev_mem=device_mem_label,
        )

    avg_loss = running_loss / max(1, running_total)
    avg_acc = running_correct / max(1, running_total)
    epoch_time_s = time.perf_counter() - epoch_start
    return avg_loss, avg_acc, epoch_time_s


## Overnight Run Manager (Multi-Run + File Logging)
Use this section to run repeated experiments unattended. It logs epoch and run summaries to `artifacts/logs/` and saves per-run checkpoints.


In [97]:
import json
from datetime import datetime

LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_GROUP_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUMMARY_CSV = LOG_DIR / "run_summary.csv"
EPOCH_METRICS_CSV = LOG_DIR / "epoch_metrics.csv"
RUN_CONFIG_JSONL = LOG_DIR / "run_configs.jsonl"


def append_df(path: Path, rows: list[dict]):
    if not rows:
        return
    df = pd.DataFrame(rows)
    write_header = not path.exists()
    df.to_csv(path, mode="a", header=write_header, index=False)


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def format_float_tag_local(x: float) -> str:
    return f"{x:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def config_tag(cfg: dict) -> str:
    parts = []
    if cfg.get("weight_decay", WEIGHT_DECAY) != DEFAULT_WEIGHT_DECAY:
        parts.append(f"decay_{format_float_tag_local(cfg.get('weight_decay', WEIGHT_DECAY))}")
    split_file = cfg.get("split_file", TRAIN_TEST_SPLIT_FILE)
    if split_file != DEFAULT_TRAIN_TEST_SPLIT_FILE:
        stem = Path(split_file).stem
        m = re.search(r"(\d{2})(\d{2})", stem)
        if m and int(m.group(1)) + int(m.group(2)) == 100:
            parts.append(f"tt_{int(m.group(1))}-{int(m.group(2))}")
        else:
            parts.append(f"tt_{stem}")
    if cfg.get("label_smoothing", LABEL_SMOOTHING) != LABEL_SMOOTHING:
        parts.append(f"ls_{cfg['label_smoothing']}")
    if cfg.get("lr", LEARNING_RATE) != LEARNING_RATE:
        parts.append(f"lr_{format_float_tag_local(cfg['lr'])}")
    if cfg.get("num_epochs", NUM_EPOCHS) != NUM_EPOCHS:
        parts.append(f"ep_{cfg['num_epochs']}")
    if cfg.get("batch_size", BATCH_SIZE) != BATCH_SIZE:
        parts.append(f"bs_{cfg['batch_size']}")
    return ("_" + "_".join(parts)) if parts else ""


In [98]:
def make_run_loaders(batch_size: int):
    # Rebuild loaders per run so batch_size can vary across configs.
    train_loader_local = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader_local = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader_local = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader_local, val_loader_local, test_loader_local


def train_stage(
    model,
    optimizer,
    criterion,
    epochs,
    run_label: str,
    stage_label: str,
    train_loader_local,
    val_loader_local,
    batch_size: int,
):
    best_val_acc = -1.0
    best_state = None
    epoch_rows = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_time = run_epoch(
            model,
            train_loader_local,
            criterion,
            optimizer=optimizer,
            split_name=f"{run_label} {stage_label} tr e{epoch:02d}",
        )
        val_loss, val_acc, val_time = run_epoch(
            model,
            val_loader_local,
            criterion,
            optimizer=None,
            split_name=f"{run_label} {stage_label} va e{epoch:02d}",
        )

        epoch_rows.append(
            {
                "run_group_id": RUN_GROUP_ID,
                "run_label": run_label,
                "stage": stage_label,
                "epoch": epoch,
                "batch_size": batch_size,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "train_time_s": train_time,
                "val_time_s": val_time,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_acc, epoch_rows


def run_one_experiment(cfg: dict, repeat_idx: int):
    run_seed = int(cfg.get("seed", SEED)) + int(repeat_idx)
    set_all_seeds(run_seed)

    run_name = cfg.get("name", "exp")
    split_file = cfg.get("split_file", TRAIN_TEST_SPLIT_FILE)
    if split_file != TRAIN_TEST_SPLIT_FILE:
        raise ValueError(
            f"Config split_file={split_file} but this notebook is currently built with "
            f"TRAIN_TEST_SPLIT_FILE={TRAIN_TEST_SPLIT_FILE}. "
            "Set TRAIN_TEST_SPLIT_FILE in the setup cell and rerun data-prep cells first."
        )
    run_label = f"{run_name}_r{repeat_idx+1:02d}"
    run_tag = config_tag(cfg)

    num_epochs = int(cfg.get("num_epochs", NUM_EPOCHS))
    lr = float(cfg.get("lr", LEARNING_RATE))
    weight_decay = float(cfg.get("weight_decay", WEIGHT_DECAY))
    label_smoothing = float(cfg.get("label_smoothing", LABEL_SMOOTHING))
    batch_size = int(cfg.get("batch_size", BATCH_SIZE))

    run_stage2 = bool(cfg.get("run_stage2", True))
    stage2_epochs = int(cfg.get("stage2_epochs", STAGE2_EPOCHS if 'STAGE2_EPOCHS' in globals() else 5))
    stage2_lr = float(cfg.get("stage2_lr", STAGE2_LR if 'STAGE2_LR' in globals() else 1e-4))
    stage2_weight_decay = float(cfg.get("stage2_weight_decay", STAGE2_WEIGHT_DECAY if 'STAGE2_WEIGHT_DECAY' in globals() else WEIGHT_DECAY))

    train_loader_local, val_loader_local, test_loader_local = make_run_loaders(batch_size=batch_size)

    run_started_at = datetime.now().isoformat(timespec="seconds")
    run_t0 = time.perf_counter()

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Stage 1: head-only
    model_s1 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    for p in model_s1.parameters():
        p.requires_grad = False
    model_s1.fc = nn.Linear(model_s1.fc.in_features, len(final_species))
    model_s1 = model_s1.to(DEVICE)

    opt_s1 = torch.optim.AdamW(model_s1.fc.parameters(), lr=lr, weight_decay=weight_decay)
    model_s1, s1_best_val_acc, s1_epoch_rows = train_stage(
        model_s1,
        opt_s1,
        criterion,
        epochs=num_epochs,
        run_label=run_label,
        stage_label="stage1",
        train_loader_local=train_loader_local,
        val_loader_local=val_loader_local,
        batch_size=batch_size,
    )

    s1_test_loss, s1_test_acc, s1_test_time = run_epoch(
        model_s1,
        test_loader_local,
        criterion,
        optimizer=None,
        split_name=f"{run_label} stage1 te",
    )

    s1_ckpt_name = f"resnet50_nabirds_head_only{run_tag}_{run_label}_{RUN_GROUP_ID}.pt"
    s1_ckpt_path = OUTPUT_DIR / s1_ckpt_name
    torch.save(model_s1.state_dict(), s1_ckpt_path)

    summary_rows = [
        {
            "run_group_id": RUN_GROUP_ID,
            "run_label": run_label,
            "run_started_at": run_started_at,
            "stage": "stage1",
            "seed": run_seed,
            "split_file": split_file,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            "lr": lr,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "best_val_acc": s1_best_val_acc,
            "test_loss": s1_test_loss,
            "test_acc": s1_test_acc,
            "test_time_s": s1_test_time,
            "run_time_s": time.perf_counter() - run_t0,
            "checkpoint_path": str(s1_ckpt_path),
            "config_json": json.dumps(cfg, sort_keys=True),
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        }
    ]

    epoch_rows = list(s1_epoch_rows)

    if run_stage2:
        # Stage 2: layer4 + fc
        model_s2 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model_s2.fc = nn.Linear(model_s2.fc.in_features, len(final_species))
        model_s2.load_state_dict(torch.load(s1_ckpt_path, map_location="cpu"))

        for p in model_s2.parameters():
            p.requires_grad = False
        for p in model_s2.layer4.parameters():
            p.requires_grad = True
        for p in model_s2.fc.parameters():
            p.requires_grad = True
        model_s2 = model_s2.to(DEVICE)

        opt_s2 = torch.optim.AdamW(
            [p for p in model_s2.parameters() if p.requires_grad],
            lr=stage2_lr,
            weight_decay=stage2_weight_decay,
        )

        model_s2, s2_best_val_acc, s2_epoch_rows = train_stage(
            model_s2,
            opt_s2,
            criterion,
            epochs=stage2_epochs,
            run_label=run_label,
            stage_label="stage2",
            train_loader_local=train_loader_local,
            val_loader_local=val_loader_local,
            batch_size=batch_size,
        )

        s2_test_loss, s2_test_acc, s2_test_time = run_epoch(
            model_s2,
            test_loader_local,
            criterion,
            optimizer=None,
            split_name=f"{run_label} stage2 te",
        )

        s2_ckpt_name = f"resnet50_nabirds_layer4_finetuned{run_tag}_{run_label}_{RUN_GROUP_ID}.pt"
        s2_ckpt_path = OUTPUT_DIR / s2_ckpt_name
        torch.save(model_s2.state_dict(), s2_ckpt_path)

        summary_rows.append(
            {
                "run_group_id": RUN_GROUP_ID,
                "run_label": run_label,
                "run_started_at": run_started_at,
                "stage": "stage2",
                "seed": run_seed,
                "split_file": split_file,
                "batch_size": batch_size,
                "num_epochs": stage2_epochs,
                "lr": stage2_lr,
                "weight_decay": stage2_weight_decay,
                "label_smoothing": label_smoothing,
                "best_val_acc": s2_best_val_acc,
                "test_loss": s2_test_loss,
                "test_acc": s2_test_acc,
                "test_time_s": s2_test_time,
                "run_time_s": time.perf_counter() - run_t0,
                "checkpoint_path": str(s2_ckpt_path),
                "config_json": json.dumps(cfg, sort_keys=True),
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            }
        )

        epoch_rows.extend(s2_epoch_rows)

    return summary_rows, epoch_rows


## Execute Run Program
Run this after all setup/data/runtime function cells have been executed.


In [99]:
print(f"Run group id: {RUN_GROUP_ID}")
print("Summary log:", RUN_SUMMARY_CSV)
print("Epoch log:  ", EPOCH_METRICS_CSV)
print("Config log: ", RUN_CONFIG_JSONL)

all_summary_rows = []
all_epoch_rows = []

for cfg in RUN_PROGRAM:
    repeats = int(cfg.get("repeats", 1))
    with open(RUN_CONFIG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps({"run_group_id": RUN_GROUP_ID, "config": cfg}, sort_keys=True) + "\n")

    for r in range(repeats):
        print(f"\n=== Running {cfg.get('name','exp')} repeat {r+1}/{repeats} ===")
        summary_rows, epoch_rows = run_one_experiment(cfg, repeat_idx=r)

        # Persist after every run so overnight interruptions still keep results.
        append_df(RUN_SUMMARY_CSV, summary_rows)
        append_df(EPOCH_METRICS_CSV, epoch_rows)

        all_summary_rows.extend(summary_rows)
        all_epoch_rows.extend(epoch_rows)

summary_df = pd.DataFrame(all_summary_rows)
if not summary_df.empty:
    display(summary_df.sort_values(["run_label", "stage"]))
else:
    print("No runs executed.")


Run group id: 20260215_220901
Summary log: artifacts/logs/run_summary.csv
Epoch log:   artifacts/logs/epoch_metrics.csv
Config log:  artifacts/logs/run_configs.jsonl

=== Running learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half repeat 1/1 ===


learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half_r01 stage1 tr e01:   0%|          | 0/328 [00:00<?, …

learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half_r01 stage1 va e01:   0%|          | 0/37 [00:00<?, ?…

learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half_r01 stage1 tr e02:   0%|          | 0/328 [00:00<?, …

learning-rate-half_label-smooth-0-05_decay-2e4_s2-lr-half_r01 stage1 va e02:   0%|          | 0/37 [00:00<?, ?…

...

learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter_r01 stage2 va e04:   0%|          | 0/37 [00:0…

learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter_r01 stage2 tr e05:   0%|          | 0/328 [00:…

learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter_r01 stage2 va e05:   0%|          | 0/37 [00:0…

learning-rate-quarter_label-smooth-0-05_decay-1e4_s2-lr-quarter_r01 stage2 te:   0%|          | 0/92 [00:00<?,…

,run_group_id,run_label,run_started_at,stage,seed,split_file,batch_size,num_epochs,lr,weight_decay,label_smoothing,best_val_acc,test_loss,test_acc,test_time_s,run_time_s,checkpoint_path,config_json,timestamp
2,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,2026-02-15T22:34:01,stage1,42,train_test_split_8020_target_species.txt,32,8,0.00050,0.0001,0.05,0.761905,1.356234,0.783608,23.870673,862.453154,artifacts/resnet50_nabirds_head_only_tt_80-20_...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T22:48:24
3,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,2026-02-15T22:34:01,stage2,42,train_test_split_8020_target_species.txt,32,5,0.00010,0.0001,0.05,0.900510,0.814973,0.916324,24.437934,1507.294283,artifacts/resnet50_nabirds_layer4_finetuned_tt...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T22:59:08
6,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,2026-02-15T23:24:25,stage1,42,train_test_split_8020_target_species.txt,32,8,0.00050,0.0001,0.05,0.761905,1.356234,0.783608,23.741818,854.548958,artifacts/resnet50_nabirds_head_only_tt_80-20_...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T23:38:40
7,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,2026-02-15T23:24:25,stage2,42,train_test_split_8020_target_species.txt,32,5,0.00010,0.0001,0.05,0.900510,0.814973,0.916324,23.739786,1488.663104,artifacts/resnet50_nabirds_layer4_finetuned_tt...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T23:49:14
0,20260215_220901,learning-rate-half_label-smooth-0-05_decay-2e4...,2026-02-15T22:09:01,stage1,42,train_test_split_8020_target_species.txt,32,8,0.00050,0.0002,0.05,0.761905,1.356253,0.783608,23.887118,857.859765,artifacts/resnet50_nabirds_head_only_decay_2e-...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T22:23:19
1,20260215_220901,learning-rate-half_label-smooth-0-05_decay-2e4...,2026-02-15T22:09:01,stage2,42,train_test_split_8020_target_species.txt,32,5,0.00010,0.0001,0.05,0.898810,0.817878,0.909808,23.868881,1500.028001,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T22:34:01
4,20260215_220901,learning-rate-half_label-smooth-0-05_decay-2e4...,2026-02-15T22:59:08,stage1,42,train_test_split_8020_target_species.txt,32,8,0.00050,0.0002,0.05,0.761905,1.356253,0.783608,24.422350,869.101465,artifacts/resnet50_nabirds_head_only_decay_2e-...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T23:13:38
5,20260215_220901,learning-rate-half_label-smooth-0-05_decay-2e4...,2026-02-15T22:59:08,stage2,42,train_test_split_8020_target_species.txt,32,5,0.00010,0.0001,0.05,0.898810,0.817878,0.909808,24.402118,1516.624661,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-15T23:24:25
10,20260215_220901,learning-rate-quarter_label-smooth-0-05_decay-...,2026-02-16T00:13:56,stage1,42,train_test_split_8020_target_species.txt,32,8,0.00025,0.0001,0.05,0.697279,1.683249,0.727366,23.771548,848.745088,artifacts/resnet50_nabirds_head_only_tt_80-20_...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-16T00:28:05
11,20260215_220901,learning-rate-quarter_label-smooth-0-05_decay-...,2026-02-16T00:13:56,stage2,42,train_test_split_8020_target_species.txt,32,5,0.00010,0.0001,0.05,0.907313,0.805153,0.914609,23.731976,1482.796932,artifacts/resnet50_nabirds_layer4_finetuned_tt...,"{""batch_size"": 32, ""label_smoothing"": 0.05, ""l...",2026-02-16T00:38:39


## Results Summary (From Log Files)
Loads overnight run logs and summarizes best runs and repeat-level stability.


In [100]:
from pathlib import Path

LOG_DIR = Path("artifacts/logs")
RUN_SUMMARY_CSV = LOG_DIR / "run_summary.csv"
EPOCH_METRICS_CSV = LOG_DIR / "epoch_metrics.csv"

if not RUN_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing {RUN_SUMMARY_CSV}. Run the Overnight Run Manager first.")

run_summary_df = pd.read_csv(RUN_SUMMARY_CSV)
run_summary_df


,run_group_id,run_label,run_started_at,stage,seed,split_file,batch_size,num_epochs,lr,weight_decay,label_smoothing,best_val_acc,test_loss,test_acc,test_time_s,run_time_s,checkpoint_path,config_json,timestamp
0,20260214_230109,overnight_base_r01,2026-02-14T23:01:09,stage1,42,train_test_split_8020_target_species.txt,64,8,0.00200,0.0002,0.00,0.779762,0.781953,0.801783,32.737414,1193.282312,artifacts/resnet50_nabirds_head_only_decay_2e-...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-14T23:21:02
1,20260214_230109,overnight_base_r01,2026-02-14T23:01:09,stage2,42,train_test_split_8020_target_species.txt,64,5,0.00010,0.0002,0.00,0.877551,0.390676,0.891289,23.791624,1869.531117,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-14T23:32:18
2,20260214_230109,weight_decay_test_1e-4_r01,2026-02-14T23:32:18,stage1,42,train_test_split_8020_target_species.txt,64,8,0.00200,0.0001,0.00,0.779762,0.781905,0.801783,24.168052,863.333484,artifacts/resnet50_nabirds_head_only_tt_80-20_...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-14T23:46:42
3,20260214_230109,weight_decay_test_1e-4_r01,2026-02-14T23:32:18,stage2,42,train_test_split_8020_target_species.txt,64,5,0.00010,0.0002,0.00,0.878401,0.394547,0.888203,26.914779,1504.399082,artifacts/resnet50_nabirds_layer4_finetuned_tt...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T00:02:22
4,20260214_230109,weight_decay_test_2e-4_r01,2026-02-15T00:02:22,stage1,42,train_test_split_8020_target_species.txt,64,8,0.00200,0.0002,0.00,0.779762,0.781953,0.801783,23.382266,881.731482,artifacts/resnet50_nabirds_head_only_decay_2e-...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T01:12:04
5,20260214_230109,weight_decay_test_2e-4_r01,2026-02-15T00:02:22,stage2,42,train_test_split_8020_target_species.txt,64,5,0.00010,0.0002,0.00,0.877551,0.390676,0.891289,23.397185,1510.770414,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T02:28:13
6,20260214_230109,weight_decay_test_4e-4_r01,2026-02-15T02:28:13,stage1,42,train_test_split_8020_target_species.txt,64,8,0.00200,0.0004,0.00,0.779762,0.782026,0.801783,25.071248,936.040760,artifacts/resnet50_nabirds_head_only_decay_4e-...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T02:43:49
7,20260214_230109,weight_decay_test_4e-4_r01,2026-02-15T02:28:13,stage2,42,train_test_split_8020_target_species.txt,64,5,0.00010,0.0002,0.00,0.880952,0.391500,0.890261,28.333594,1734.183988,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T02:57:07
8,20260214_230109,batch_num_test_64_r01,2026-02-15T02:57:07,stage1,42,train_test_split_8020_target_species.txt,64,8,0.00400,0.0002,0.00,0.785714,0.696515,0.806927,28.247991,1042.359968,artifacts/resnet50_nabirds_head_only_decay_2e-...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T03:14:30
9,20260214_230109,batch_num_test_64_r01,2026-02-15T02:57:07,stage2,42,train_test_split_8020_target_species.txt,64,5,0.00010,0.0002,0.00,0.875000,0.445008,0.875857,30.797974,1865.950897,artifacts/resnet50_nabirds_layer4_finetuned_de...,"{""batch_size"": 64, ""label_smoothing"": 0.0, ""lr...",2026-02-15T03:28:13


In [101]:
# Optional: filter to a specific run group (set to None to use all logs)
RUN_GROUP_FILTER = None

summary = run_summary_df.copy()
if RUN_GROUP_FILTER is not None:
    summary = summary[summary["run_group_id"] == RUN_GROUP_FILTER].copy()

print("Rows in summary:", len(summary))
print("Run groups:", summary["run_group_id"].nunique())
print("Stages:", sorted(summary["stage"].dropna().unique().tolist()))

# Best runs by test accuracy per stage
best_by_stage = (
    summary.sort_values(["stage", "test_acc"], ascending=[True, False])
    .groupby("stage", as_index=False)
    .head(10)
    .reset_index(drop=True)
)

display(best_by_stage[[
    "run_group_id", "run_label", "stage", "test_acc", "test_loss", "best_val_acc",
    "run_time_s", "checkpoint_path", "split_file", "label_smoothing", "lr", "weight_decay"
]])


Rows in summary: 40
Run groups: 5
Stages: ['stage1', 'stage2']


,run_group_id,run_label,stage,test_acc,test_loss,best_val_acc,run_time_s,checkpoint_path,split_file,label_smoothing,lr,weight_decay
0,20260214_230109,batch_num_test_128_r01,stage1,0.809671,0.686793,0.785714,891.468965,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.00,0.0080,0.0002
1,20260215_183406,overnight_base_r01,stage1,0.807613,1.259557,0.784014,858.165723,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.05,0.0020,0.0002
2,20260214_230109,batch_num_test_64_r01,stage1,0.806927,0.696515,0.785714,1042.359968,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.00,0.0040,0.0002
3,20260214_230109,label_smoothing_test_0_10_r01,stage1,0.806241,1.550439,0.783163,866.229942,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.10,0.0020,0.0002
4,20260214_230109,overnight_base_r01,stage1,0.801783,0.781953,0.779762,1193.282312,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.00,0.0020,0.0002
5,20260214_230109,weight_decay_test_1e-4_r01,stage1,0.801783,0.781905,0.779762,863.333484,artifacts/resnet50_nabirds_head_only_tt_80-20_...,train_test_split_8020_target_species.txt,0.00,0.0020,0.0001
6,20260214_230109,weight_decay_test_2e-4_r01,stage1,0.801783,0.781953,0.779762,881.731482,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.00,0.0020,0.0002
7,20260214_230109,weight_decay_test_4e-4_r01,stage1,0.801783,0.782026,0.779762,936.040760,artifacts/resnet50_nabirds_head_only_decay_4e-...,train_test_split_8020_target_species.txt,0.00,0.0020,0.0004
8,20260214_230109,label_smoothing_test_0_05_r01,stage1,0.799726,1.201154,0.786565,848.643954,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.05,0.0020,0.0002
9,20260215_211701,learning_rate_half_label_smooth_0_05_decay_2e4...,stage1,0.783608,1.356253,0.761905,868.609563,artifacts/resnet50_nabirds_head_only_decay_2e-...,train_test_split_8020_target_species.txt,0.05,0.0005,0.0002


In [102]:
# Repeat-level stability: mean/std across repeats (grouped by config + stage)
def safe_col(df, col, default=""):
    return df[col] if col in df.columns else default

agg_cols = ["stage", "split_file", "label_smoothing", "lr", "weight_decay"]
for c in agg_cols:
    if c not in summary.columns:
        summary[c] = ""

repeat_stats = (
    summary.groupby(agg_cols, dropna=False)
    .agg(
        runs=("run_label", "count"),
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        test_loss_mean=("test_loss", "mean"),
        best_val_acc_mean=("best_val_acc", "mean"),
        run_time_mean_s=("run_time_s", "mean"),
    )
    .reset_index()
    .sort_values(["stage", "test_acc_mean"], ascending=[True, False])
)

display(repeat_stats)


,stage,split_file,label_smoothing,lr,weight_decay,runs,test_acc_mean,test_acc_std,test_loss_mean,best_val_acc_mean,run_time_mean_s
5,stage1,train_test_split_8020_target_species.txt,0.00,0.00800,0.0002,1,0.809671,NaN,0.686793,0.785714,891.468965
4,stage1,train_test_split_8020_target_species.txt,0.00,0.00400,0.0002,1,0.806927,NaN,0.696515,0.785714,1042.359968
11,stage1,train_test_split_8020_target_species.txt,0.10,0.00200,0.0002,1,0.806241,NaN,1.550439,0.783163,866.229942
10,stage1,train_test_split_8020_target_species.txt,0.05,0.00200,0.0002,2,0.803669,0.005577,1.230356,0.785289,853.404838
1,stage1,train_test_split_8020_target_species.txt,0.00,0.00200,0.0001,1,0.801783,NaN,0.781905,0.779762,863.333484
2,stage1,train_test_split_8020_target_species.txt,0.00,0.00200,0.0002,2,0.801783,0.000000,0.781953,0.779762,1037.506897
3,stage1,train_test_split_8020_target_species.txt,0.00,0.00200,0.0004,1,0.801783,NaN,0.782026,0.779762,936.040760
8,stage1,train_test_split_8020_target_species.txt,0.05,0.00050,0.0001,3,0.783608,0.000000,1.356234,0.761905,859.468964
9,stage1,train_test_split_8020_target_species.txt,0.05,0.00050,0.0002,3,0.783608,0.000000,1.356253,0.761905,865.190264
0,stage1,train_test_split_8020_target_species.txt,0.00,0.00050,0.0001,1,0.779150,NaN,0.999194,0.750000,859.538959


In [103]:
# Optional epoch-level view
if EPOCH_METRICS_CSV.exists():
    epoch_df = pd.read_csv(EPOCH_METRICS_CSV)
    if RUN_GROUP_FILTER is not None:
        epoch_df = epoch_df[epoch_df["run_group_id"] == RUN_GROUP_FILTER].copy()

    # Last-epoch snapshot per run/stage
    epoch_last = (
        epoch_df.sort_values(["run_label", "stage", "epoch"])
        .groupby(["run_label", "stage"], as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )
    display(epoch_last[["run_group_id", "run_label", "stage", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]])
else:
    print(f"Epoch metrics file not found: {EPOCH_METRICS_CSV}")


,run_group_id,run_label,stage,epoch,train_loss,train_acc,val_loss,val_acc
0,20260214_230109,batch_num_test_128_r01,stage1,8,0.069148,0.994565,0.746583,0.785714
1,20260214_230109,batch_num_test_128_r01,stage2,5,0.001274,1.000000,0.626110,0.857993
2,20260214_230109,batch_num_test_64_r01,stage1,8,0.114803,0.987985,0.748681,0.782313
3,20260214_230109,batch_num_test_64_r01,stage2,5,0.003025,0.999809,0.543656,0.871599
4,20260214_230109,label_smoothing_test_0_05_r01,stage1,8,0.713243,0.977115,1.221704,0.783163
5,20260214_230109,label_smoothing_test_0_05_r01,stage2,5,0.489490,0.999809,0.901113,0.889456
6,20260214_230109,label_smoothing_test_0_10_r01,stage1,8,1.092139,0.978736,1.575821,0.781463
7,20260214_230109,label_smoothing_test_0_10_r01,stage2,5,0.859462,0.999905,1.268319,0.890306
8,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,stage1,8,1.084048,0.893487,1.404432,0.761905
9,20260215_220901,learning-rate-half_label-smooth-0-05_decay-1e4...,stage2,5,0.498112,0.999714,0.845399,0.900510
